### Step 0

Find an NYC dataset with a borough column.

**What's the URL of your dataset?**

I used these NYC Open Data datasets (both include a Borough column):

1.
https://data.cityofnewyork.us/Social-Services/High-School-Equivalency-HSE-Diplomas-Awarded-to-NY/rc2t-8fid/about_data

2.
https://data.cityofnewyork.us/Social-Services/NYC-Business-Solutions-for-NYCHA-Residents-by-Boro/jege-pgbz/about_data

### Step 1

[Load it into Jupyter.](https://python-public-policy.afeld.me/en/columbia/assignments/open_ended.html#storing-data)

In [1]:
import pandas as pd

hse_df = pd.read_csv('/content/High_School_Equivalency_(HSE)_Diplomas_Awarded_to_NYCHA_Residents_by_Borough_–_Local_Law_163_20260203.csv')
display(hse_df.head())

,Year,Borough,Number of public housing residents who earned HSE diplomas
0,2024,Bronx,20
1,2024,Brooklyn,29
2,2024,Manhattan,15
3,2024,Queens,<5
4,2024,Staten Island,6


### Step 2

Open the [Population by Borough](https://data.cityofnewyork.us/City-Government/New-York-City-Population-by-Borough-1950-2040/xywu-7bv9) dataset and [load it into Jupyter](https://python-public-policy.afeld.me/en/columbia/assignments/open_ended.html#storing-data).

In [3]:
import pandas as pd
import requests
import io

population_url = 'https://data.cityofnewyork.us/api/views/xywu-7bv9/rows.csv?accessType=DOWNLOAD'
response = requests.get(population_url)
population_df = pd.read_csv(io.StringIO(response.text))
display(population_df.head())

,Age Group,Borough,1950,1950 - Boro share of NYC total,1960,1960 - Boro share of NYC total,1970,1970 - Boro share of NYC total,1980,1980 - Boro share of NYC total,...,2000,2000 - Boro share of NYC total,2010,2010 - Boro share of NYC total,2020,2020 - Boro share of NYC total,2030,2030 - Boro share of NYC total,2040,2040 - Boro share of NYC total
0,Total Population,NYC Total,7891957,100.00,7781984,100.00,7894862,100.00,7071639,100.00,...,8008278,100.00,8242624,100.00,8550971,100.00,8821027,100.00,9025145,100.00
1,Total Population,Bronx,1451277,18.39,1424815,18.31,1471701,18.64,1168972,16.53,...,1332650,16.64,1385108,16.80,1446788,16.92,1518998,17.22,1579245,17.50
2,Total Population,Brooklyn,2738175,34.70,2627319,33.76,2602012,32.96,2230936,31.55,...,2465326,30.78,2552911,30.97,2648452,30.97,2754009,31.22,2840525,31.47
3,Total Population,Manhattan,1960101,24.84,1698281,21.82,1539233,19.50,1428285,20.20,...,1537195,19.20,1585873,19.24,1638281,19.16,1676720,19.01,1691617,18.74
4,Total Population,Queens,1550849,19.65,1809578,23.25,1986473,25.16,1891325,26.75,...,2229379,27.84,2250002,27.30,2330295,27.25,2373551,26.91,2412649,26.73


### Step 3

Use [`merge()`](https://pandas.pydata.org/docs/user_guide/merging.html#merge) to combine the two, and output the resulting table.

In [4]:
# To merge the datasets, we need to ensure the 'Borough' columns have consistent values and find a common year for merging.
# Let's inspect the unique values and types of 'Borough' in both dataframes first.
print("Unique Boroughs in hse_df:", hse_df['Borough'].unique())
print("Unique Boroughs in population_df:", population_df['Borough'].unique())

# Let's also look at the years available in hse_df to pick a common year for population data
print("Years in hse_df:", hse_df['Year'].unique())

# The population data is in wide format, so we need to melt it to a long format first
# We'll select relevant columns for melting. The 'Age Group' is 'Total Population' for all rows we care about.
population_long_df = population_df[population_df['Age Group'] == 'Total Population'].melt(
    id_vars=['Age Group', 'Borough'],
    var_name='Year_Raw',
    value_name='Population'
)

# Extract the year from 'Year_Raw' and convert to integer
population_long_df['Year'] = population_long_df['Year_Raw'].str.extract(r'(\d{4})').astype(int)

# Filter out rows where Year_Raw contains 'Boro share' since we only want population numbers
population_long_df = population_long_df[~population_long_df['Year_Raw'].str.contains('Boro share')]

# Clean 'Borough' names in population_long_df to match hse_df
# It seems 'Staten Island' in population_df is 'Staten Island' in hse_df. 'Bronx', 'Brooklyn', 'Manhattan', 'Queens' are also consistent.
# We need to handle 'NYC Total' if it causes issues, but for borough-level merge it won't be used.

# Ensure both 'Borough' columns are of string type for consistent merging
hse_df['Borough'] = hse_df['Borough'].astype(str)
population_long_df['Borough'] = population_long_df['Borough'].astype(str)

# Merge the datasets on 'Borough' and 'Year'
# We will perform a left merge from hse_df to include all HSE data and add population where available.
merged_df = pd.merge(
    hse_df,
    population_long_df[['Borough', 'Year', 'Population']],
    on=['Borough', 'Year'],
    how='left'
)

display(merged_df.head())

Unique Boroughs in hse_df: ['Bronx' 'Brooklyn' 'Manhattan' 'Queens' 'Staten Island']
Unique Boroughs in population_df: ['NYC Total' '   Bronx' '   Brooklyn' '   Manhattan' '   Queens'
 '   Staten Island']
Years in hse_df: [2024 2023 2022 2021 2020 2019 2018 2017]


,Year,Borough,Number of public housing residents who earned HSE diplomas,Population
0,2024,Bronx,20,NaN
1,2024,Brooklyn,29,NaN
2,2024,Manhattan,15,NaN
3,2024,Queens,<5,NaN
4,2024,Staten Island,6,NaN


<details>
<summary><strong>Hint</strong></summary>
Having trouble merging the datasets? Try looking at the unique values in the columns you're trying to merge on.
</details>

### Step 4

Using the two datasets above, use pandas to produce an aggregate per-capita statistic by borough.

The dataset you chose before may not work for this. That's fine, pick another.

<details>
<summary><strong>Hint</strong></summary>
You're creating a "number of [thing] per capita by borough" table.

1. Do a [`groupby()`](https://pandas.pydata.org/docs/user_guide/groupby.html) on the original dataset.
1. Join with the populations by borough.
1. Compute the per-capita values as a new column.
</details>

In [8]:
# Clean 'Borough' names in population_long_df (strip leading/trailing spaces)
population_long_df['Borough'] = population_long_df['Borough'].str.strip()

# Filter out 'NYC Total' from population_long_df as it's not a specific borough
population_long_df = population_long_df[population_long_df['Borough'] != 'NYC Total']

# Handle the '<5' string and other potential non-numeric values in 'Number of public housing residents who earned HSE diplomas' column
# Replace any non-digit characters with an empty string, then convert to numeric. Coerce errors to NaN and fill with 0.
hse_df['Number of public housing residents who earned HSE diplomas_cleaned'] = \
    hse_df['Number of public housing residents who earned HSE diplomas'].astype(str).str.replace(r'[^0-9]', '', regex=True)
# Convert to numeric, coercing errors to NaN, then fill NaN with 0 and convert to int
hse_df['Number of public housing residents who earned HSE diplomas_cleaned'] = pd.to_numeric(
    hse_df['Number of public housing residents who earned HSE diplomas_cleaned'], errors='coerce'
).fillna(0).astype(int)

# Aggregate HSE diplomas by Borough and Year
aggregated_hse_df = hse_df.groupby(['Borough', 'Year'])['Number of public housing residents who earned HSE diplomas_cleaned'].sum().reset_index()
aggregated_hse_df.rename(columns={'Number of public housing residents who earned HSE diplomas_cleaned': 'Total HSE Diplomas'}, inplace=True)

# Merge aggregated HSE diplomas with population data
# Use an inner merge to only include boroughs and years present in both datasets
per_capita_df = pd.merge(
    aggregated_hse_df,
    population_long_df[['Borough', 'Year', 'Population']],
    on=['Borough', 'Year'],
    how='inner'
)

# Calculate per-capita HSE diplomas
# To avoid division by zero, we can replace 0 populations with NaN or a very small number,
# or filter them out. For now, let's proceed and check for NaNs.
per_capita_df['HSE Diplomas Per Capita'] = per_capita_df['Total HSE Diplomas'] / per_capita_df['Population']

# Display the resulting table, sorted for better readability
display(per_capita_df.sort_values(by=['Year', 'Borough']).head(10))

,Borough,Year,Total HSE Diplomas,Population,HSE Diplomas Per Capita
0,Bronx,2020,34,1446788.0,0.000024
1,Brooklyn,2020,38,2648452.0,0.000014
2,Manhattan,2020,12,1638281.0,0.000007
3,Queens,2020,5,2330295.0,0.000002
4,Staten Island,2020,11,487155.0,0.000023


Now [turn in the assignment](https://python-public-policy.afeld.me/en/columbia/assignments.html).
